# PixCell Multi-Channel ControlNet — Colab Training (Stages 0–2)

This notebook runs the computationally expensive stages of the PixCell ControlNet pipeline on Google Colab:

| Stage | Script | What it does |
|-------|--------|--------------|
| **0** | `stage0_setup.py` | Download pretrained models (PixCell-256, ControlNet, UNI-2h, SD3.5 VAE) |
| **1** | `stage1_extract_features.py` | Extract UNI embeddings + VAE latents from H&E and cell-mask images |
| **2** | `stage2_train.py` | Train ControlNet + TME module on paired ORION-CRC data |

Stage 3 (inference) is lightweight and can run locally.

## Environment Setup

Import core libraries and set up the Colab environment.

In [1]:
import os
import numpy as np
import torch
from IPython.display import clear_output
from google.colab import userdata

clear_output()

## Clone Repository & Install Dependencies

Clone the PixCell repo, check out the ControlNet branch, and install Python packages.

In [2]:
%cd /content
!git clone https://github.com/pohaoc2/PixCell.git
%cd /content/PixCell
!git checkout main
!pip install -r requirements.txt
!pip install mmcv==1.7.0
clear_output()

## Configure AWS & HuggingFace Credentials

Set AWS credentials (for S3 data access) and HuggingFace token (for gated model downloads).
These should be stored in Colab's **Secrets** tab (`userdata`).

In [3]:
!pip install awscli
clear_output()

os.environ["AWS_ACCESS_KEY_ID"] = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"] = "us-west-2"
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

## Download ORION-CRC Data from S3

Download the paired experimental dataset (H&E tiles + multi-channel TME maps) into `data/orion-crc/`.
The dataset contains:
- `he/` — H&E image tiles (256×256 PNG)
- `exp_channels/` — per-channel TME maps (cell_mask, cell_type_*, cell_state_*, vasculature, oxygen, glucose)

> **Note:** Update `S3_DATA_PATH` with the actual S3 URI before running.

In [ ]:
%cd /content/PixCell

S3_DATA_PATH = "s3://bagherilab-working/pohao/share_space/orion-crc33.zip"
#S3_DATA_PATH = "s3://bagherilab-working/pohao/share_space/test-orion-crc33.zip"
!aws s3 cp {S3_DATA_PATH} ./data/orion-crc33.zip
!unzip ./data/orion-crc33.zip -d ./data/

clear_output()

In [ ]:
print("H&E tiles:", len(os.listdir("data/orion-crc33/he")))
print("Channel dirs:", os.listdir("data/orion-crc33/exp_channels"))

## Stage 0 — Download Pretrained Models

Download all four pretrained model components into `pretrained_models/`:
- **PixCell-256** transformer (frozen base model)
- **PixCell-256 ControlNet** (initialization weights)
- **UNI-2h** (histopathology feature extractor)
- **SD 3.5 VAE** (image encoder/decoder)

Requires `HF_TOKEN` for gated model access.

In [5]:
%cd /content/PixCell
!python stage0_setup.py --token {os.environ['HF_TOKEN']}
clear_output()
print("Pretrained models:")
!ls pretrained_models/

Pretrained models:
pixcell-256  pixcell-256-controlnet  sd-3.5-vae  uni-2h


## Stage 1 — Extract Features (Pass 1: H&E Images)

Extract **UNI-2h embeddings** (`*_uni.npy`) and **SD3.5 VAE latents** (`*_sd3_vae.npy`) from H&E tiles.
These are cached once and loaded at every training step.

In [ ]:
%cd /content/PixCell
!python stage1_extract_features.py \
    --image-dir  ./data/orion-crc33/he \
    --output-dir ./data/orion-crc33/features

## Stage 1 — Extract Features (Pass 2: Cell Masks)

Extract **VAE latents for cell_mask images** (`*_mask_sd3_vae.npy`).
These are used as additional ControlNet conditioning during training. UNI extraction is skipped for masks.

In [ ]:
%cd /content/PixCell
!python stage1_extract_features.py \
    --image-dir   ./data/orion-crc33/exp_channels/cell_mask \
    --output-dir  ./data/orion-crc33/vae_features \
    --vae-prefix  mask_sd3_vae \
    --skip-uni

## Stage 1 — Build HDF5 Metadata Index

Create the HDF5 index file that maps tile IDs to their channel/feature paths.
This index is loaded by the dataloader during training.

In [ ]:
from diffusion.data.datasets.paired_exp_controlnet_dataset import build_exp_index

build_exp_index(
    exp_channels_dir="data/orion-crc33/exp_channels",
    output_path="data/orion-crc33/metadata/exp_index.hdf5",
)

In [ ]:
%cd /content/PixCell/
!mkdir -p checkpoints/pixcell_controlnet_exp/checkpoints
%cd /content/PixCell/checkpoints/pixcell_controlnet_exp/checkpoints
!aws s3 cp s3://bagherilab-working/pohao/share_space/checkpoints/tme_module.pth tme_module.pth
!aws s3 cp s3://bagherilab-working/pohao/share_space/checkpoints/controlnet_epoch_30_step_4890.pth controlnet_epoch_30_step_4890.pth

## Stage 2 — Train A2/A3 Design-Justification Ablations

Use this section for the A2/A3 ablation runs. The setup/data cells above are unchanged.

Run the launcher cell once per training run, changing `VARIANT`, `BASE_CONFIG`, `SEED`, `EPOCHS`, and `RUN_NAME`.

Required new runs:

| Ablation | `VARIANT` | `BASE_CONFIG` | Seeds | `EPOCHS` | Runs |
|---|---|---|---|---:|---:|
| A2 short proxy | `a2_bypass` | `configs/config_controlnet_exp_a2_bypass.py` | 1,2,3,4,5 | 5 | 5 |
| A2 full headline | `a2_bypass` | `configs/config_controlnet_exp_a2_bypass.py` | 42 | 20 | 1 |
| A3 short proxy | `a3_no_zero_init` | `configs/config_controlnet_exp_a3_no_zero_init.py` | 1,2,3,4,5 | 5 | 5 |
| A3 full headline | `a3_no_zero_init` | `configs/config_controlnet_exp_a3_no_zero_init.py` | 42 | 20 | 1 |

That is **12 new training runs**.

Optional A3 production-True reference logs: if existing production checkpoints do not have `train_log.jsonl`, run short-proxy True reference seeds with `VARIANT = "a3_zero_init_true"` and `BASE_CONFIG = "configs/config_controlnet_exp.py"`. Five seeds is best for the A3 mean/std loss curve; one seed is the minimum fallback.


In [ ]:
!pip install accelerate
!accelerate config default
clear_output()


In [ ]:
%cd /content/PixCell
!git pull origin main

from pathlib import Path
import re
import subprocess

# Change these values for each run.
# A2 full-headline example:
VARIANT = "a2_bypass"
BASE_CONFIG = "configs/config_controlnet_exp_a2_bypass.py"
SEED = 42
EPOCHS = 20
RUN_NAME = "full_seed_42"

CONFIG = f"/content/PixCell/configs/tmp_{VARIANT}_{RUN_NAME}.py"
WORK_DIR = f"/content/PixCell/checkpoints/pixcell_controlnet_exp_{VARIANT}/{RUN_NAME}"
S3_BASE = f"s3://bagherilab-working/pohao/share_space/a2_a3/{VARIANT}/{RUN_NAME}"

base_text = Path(BASE_CONFIG).read_text()
if re.search(r"^num_epochs\s*=\s*\d+", base_text, flags=re.MULTILINE):
    base_text = re.sub(r"^num_epochs\s*=\s*\d+", f"num_epochs = {EPOCHS}", base_text, flags=re.MULTILINE)
else:
    base_text += f"
num_epochs = {EPOCHS}
"
Path(CONFIG).write_text(base_text)

print("Training configuration")
print("  VARIANT:", VARIANT)
print("  CONFIG:", CONFIG)
print("  SEED:", SEED)
print("  EPOCHS:", EPOCHS)
print("  RUN_NAME:", RUN_NAME)
print("  WORK_DIR:", WORK_DIR)
print("  S3_BASE:", S3_BASE)

cmd = [
    "accelerate", "launch", "stage2_train.py", CONFIG,
    "--work-dir", WORK_DIR,
    "--seed", str(SEED),
]
print("Running:", " ".join(cmd))
result = subprocess.run(cmd)
if result.returncode != 0:
    raise RuntimeError(
        "Training failed. Scroll up to the first Python traceback above this message. "
        "Upload is intentionally skipped because train_log.jsonl/checkpoints were not created."
    )

train_log = Path(WORK_DIR) / "train_log.jsonl"
ckpt_dir = Path(WORK_DIR) / "checkpoints"
if not train_log.exists():
    raise FileNotFoundError(f"Training finished but missing {train_log}")
if not ckpt_dir.exists():
    raise FileNotFoundError(f"Training finished but missing {ckpt_dir}")

subprocess.run(["aws", "s3", "cp", str(train_log), f"{S3_BASE}/train_log.jsonl"], check=True)
subprocess.run(["aws", "s3", "cp", str(ckpt_dir) + "/", f"{S3_BASE}/checkpoint/", "--recursive", "--quiet"], check=True)
print("Upload complete:", S3_BASE)


### Launcher Examples

For A2 short proxy, run the launcher cell five times with:

```python
VARIANT = "a2_bypass"
BASE_CONFIG = "configs/config_controlnet_exp_a2_bypass.py"
SEED = 1  # then 2, 3, 4, 5
EPOCHS = 5
RUN_NAME = f"seed_{SEED}"
```

For A2 full headline:

```python
VARIANT = "a2_bypass"
BASE_CONFIG = "configs/config_controlnet_exp_a2_bypass.py"
SEED = 42
EPOCHS = 20
RUN_NAME = "full_seed_42"
```

For A3 short proxy, run five times with:

```python
VARIANT = "a3_no_zero_init"
BASE_CONFIG = "configs/config_controlnet_exp_a3_no_zero_init.py"
SEED = 1  # then 2, 3, 4, 5
EPOCHS = 5
RUN_NAME = f"seed_{SEED}"
```

For A3 full headline:

```python
VARIANT = "a3_no_zero_init"
BASE_CONFIG = "configs/config_controlnet_exp_a3_no_zero_init.py"
SEED = 42
EPOCHS = 20
RUN_NAME = "full_seed_42"
```


## Stage 2 — Train A1 Architectural Variants (A6 Deferred)

Use this section for the A1 design-justification runs only. Unlike A2/A3, A1 is not a single-flag perturbation: it requires alternative model and config files.

Current repo status:
- A1 and A6 are specified in `docs/superpowers/specs/2026-04-26-A1-A6-architectural-variants-design.md`.
- A1 concat and A1 per-channel module/config files are implemented in this checkout.
- The launcher below validates that the required A1 files are present before starting a run.
- The launcher below loops over `CONCAT_RUNS` and `PER_CHANNEL_RUNS`; edit those lists to change the seed/epoch matrix.
- A6 is intentionally deferred here and remains TODO; do not schedule A6 runs from this notebook until the architecture work is explicitly prioritized.

Recommended A1-only runs:

| Variant | `VARIANT` | `BASE_CONFIG` | Seeds | `EPOCHS` | Runs |
|---|---|---|---|---:|---:|
| A1 short proxy concat | `a1_concat` | `configs/config_controlnet_exp_a1_concat.py` | 1,2,3 | 5 | 3 |
| A1 full headline concat | `a1_concat` | `configs/config_controlnet_exp_a1_concat.py` | 42 | 20 | 1 |
| A1 short proxy per-channel | `a1_per_channel` | `configs/config_controlnet_exp_a1_per_channel.py` | 1,2,3 | 5 | 3 |
| A1 full headline per-channel | `a1_per_channel` | `configs/config_controlnet_exp_a1_per_channel.py` | 42 | 20 | 1 |

The A1 production row reuses the existing production checkpoints from `configs/config_controlnet_exp.py`; no extra A1-production training run is needed.

In [ ]:
%cd /content/PixCell
!git pull origin main

from pathlib import Path
import re

IMPLEMENTATION_FILES = {
    "a1_concat": [
        "configs/config_controlnet_exp_a1_concat.py",
        "diffusion/model/nets/concat_controlnet.py",
    ],
    "a1_per_channel": [
        "configs/config_controlnet_exp_a1_per_channel.py",
        "diffusion/model/nets/per_channel_tme.py",
    ],
}

CONCAT_RUNS = [
    {"seed": 1, "epochs": 5, "run_name": "seed_1"},
    {"seed": 2, "epochs": 5, "run_name": "seed_2"},
    {"seed": 3, "epochs": 5, "run_name": "seed_3"},
    {"seed": 42, "epochs": 20, "run_name": "full_seed_42"},
]

PER_CHANNEL_RUNS = [
    {"seed": 1, "epochs": 5, "run_name": "seed_1"},
    {"seed": 2, "epochs": 5, "run_name": "seed_2"},
    {"seed": 3, "epochs": 5, "run_name": "seed_3"},
    {"seed": 42, "epochs": 20, "run_name": "full_seed_42"},
]

def prepare_a1_run(variant, base_config, seed, epochs, run_name):
    if variant not in IMPLEMENTATION_FILES:
        raise ValueError(f"VARIANT must be one of {sorted(IMPLEMENTATION_FILES)}")

    expected_config, *extra_files = IMPLEMENTATION_FILES[variant]
    if base_config != expected_config:
        raise ValueError(
            f"BASE_CONFIG for {variant!r} should be {expected_config!r}, got {base_config!r}"
        )

    required_files = [expected_config, *extra_files]
    missing = [path for path in required_files if not Path(path).exists()]
    if missing:
        missing_text = "\n".join(f"  - {path}" for path in missing)
        raise FileNotFoundError(
            "A1 is not implemented in this checkout yet. Missing required files:\n"
            f"{missing_text}\n\n"
            "Add the A1 module/config implementation before running this cell."
        )

    config = f"/content/PixCell/configs/tmp_{variant}_{run_name}.py"
    work_dir = f"/content/PixCell/checkpoints/pixcell_controlnet_exp_{variant}/{run_name}"
    s3_base = f"s3://bagherilab-working/pohao/share_space/a1_tme_design/{variant}/{run_name}"

    base_text = Path(base_config).read_text()
    if re.search(r"^num_epochs\s*=\s*\d+", base_text, flags=re.MULTILINE):
        base_text = re.sub(
            r"^num_epochs\s*=\s*\d+",
            f"num_epochs = {epochs}",
            base_text,
            flags=re.MULTILINE,
        )
    else:
        base_text += f"\nnum_epochs = {epochs}\n"
    Path(config).write_text(base_text)
    return config, work_dir, s3_base

for run in CONCAT_RUNS:
    VARIANT = "a1_concat"
    BASE_CONFIG = "configs/config_controlnet_exp_a1_concat.py"
    SEED = run["seed"]
    EPOCHS = run["epochs"]
    RUN_NAME = run["run_name"]
    CONFIG, WORK_DIR, S3_BASE = prepare_a1_run(VARIANT, BASE_CONFIG, SEED, EPOCHS, RUN_NAME)

    print("A1 training configuration")
    print("  VARIANT:", VARIANT)
    print("  CONFIG:", CONFIG)
    print("  SEED:", SEED)
    print("  EPOCHS:", EPOCHS)
    print("  RUN_NAME:", RUN_NAME)
    print("  WORK_DIR:", WORK_DIR)
    print("  S3_BASE:", S3_BASE)
    print("Running:", f"accelerate launch stage2_train.py {CONFIG} --work-dir {WORK_DIR} --seed {SEED}")
    !accelerate launch stage2_train.py {CONFIG} --work-dir {WORK_DIR} --seed {SEED}

    train_log = Path(WORK_DIR) / "train_log.jsonl"
    ckpt_dir = Path(WORK_DIR) / "checkpoints"
    if not train_log.exists():
        raise FileNotFoundError(f"Training finished but missing {train_log}")
    if not ckpt_dir.exists():
        raise FileNotFoundError(f"Training finished but missing {ckpt_dir}")

    !aws s3 cp {train_log} {S3_BASE}/train_log.jsonl
    !aws s3 cp {str(ckpt_dir)}/ {S3_BASE}/checkpoint/ --recursive --quiet
    print("Upload complete:", S3_BASE)
    print()

for run in PER_CHANNEL_RUNS:
    VARIANT = "a1_per_channel"
    BASE_CONFIG = "configs/config_controlnet_exp_a1_per_channel.py"
    SEED = run["seed"]
    EPOCHS = run["epochs"]
    RUN_NAME = run["run_name"]
    CONFIG, WORK_DIR, S3_BASE = prepare_a1_run(VARIANT, BASE_CONFIG, SEED, EPOCHS, RUN_NAME)

    print("A1 training configuration")
    print("  VARIANT:", VARIANT)
    print("  CONFIG:", CONFIG)
    print("  SEED:", SEED)
    print("  EPOCHS:", EPOCHS)
    print("  RUN_NAME:", RUN_NAME)
    print("  WORK_DIR:", WORK_DIR)
    print("  S3_BASE:", S3_BASE)
    print("Running:", f"accelerate launch stage2_train.py {CONFIG} --work-dir {WORK_DIR} --seed {SEED}")
    !accelerate launch stage2_train.py {CONFIG} --work-dir {WORK_DIR} --seed {SEED}

    train_log = Path(WORK_DIR) / "train_log.jsonl"
    ckpt_dir = Path(WORK_DIR) / "checkpoints"
    if not train_log.exists():
        raise FileNotFoundError(f"Training finished but missing {train_log}")
    if not ckpt_dir.exists():
        raise FileNotFoundError(f"Training finished but missing {ckpt_dir}")

    !aws s3 cp {train_log} {S3_BASE}/train_log.jsonl
    !aws s3 cp {str(ckpt_dir)}/ {S3_BASE}/checkpoint/ --recursive --quiet
    print("Upload complete:", S3_BASE)
    print()

## Disconnect Runtime

Release the Colab GPU once training and upload are complete.

In [ ]:
from google.colab import runtime
runtime.unassign()